# 基于双塔模型的向量化召回系统

**项目描述**：基于双塔模型分别学习用户向量和物品向量，通过向量相似度完成 U2I Top-K 召回；对比 **BCE** 与 **InfoNCE** 两种损失函数，分析不同损失函数对召回效果的影响。
**技术栈**：Python + PyTorch + Two-Tower + Transformer Encoder + FAISS 向量索引 + AUC / Precision / Recall 指标评估

- 基于推荐系统召回链路实现 Two-Tower 向量召回模型，分别构建 User Tower 和 Item Tower，将用户与物品映射到统一 embedding 空间，通过相似度计算完成 U2I Top-K 候选召回。
- 用户塔融合 user_id、gender、age 及历史行为序列等特征，物品塔融合 item_id 与视频类目特征，保证召回阶段用户侧与物品侧表达解耦，适配离线 embedding 生成与在线 ANN 检索场景。
- 使用 Transformer 对用户历史行为序列进行建模，并结合 padding mask 处理不等长序列，相比仅使用 user_id 的静态表示，能够更充分刻画用户动态兴趣偏好。
- 对比 **BCE** 与 **InfoNCE** 两种损失函数：BCE 更适合点式点击监督、收敛稳定；InfoNCE 强化正负样本区分，适合学习更有判别力的召回向量，但对 batch 中正样本数量更敏感。
- 集成 FAISS IVF ANN 向量检索，在大规模物品集合下提升召回效率；同时保留 Exact 全量打分作为基线，便于定量比较召回质量与检索速度。
- 完成数据清洗、Label Encoding、历史行为序列构造、模型训练与候选物品批量打分流程，验证集 AUC 最高约 **0.742**，形成从训练到召回的完整闭环。

In [3]:
# ===== 1) 依赖导入与全局配置 =====
import json
import os
import sys
from typing import Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    faiss = None
    FAISS_AVAILABLE = False

# 让 notebook 在任意工作目录下都能导入项目模块
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from models.two_tower_recall import build_two_tower_model

# 统一配置区：训练和召回都从这里读取参数
CONFIG = {
    "data_path": os.path.join(project_root, "data/ctr_data_500k.csv"),
    "max_seq_len": 10, # 用户历史行为序列的最大长度
    "batch_size": 128, # 批次大小
    "num_epochs": 6, # 训练轮数,把训练集完整跑一遍叫做1个epoch
    "learning_rate": 0.001, # 学习率，控制参数更新步长，太大容易震荡，太小收敛慢
    "weight_decay": 1e-5, # 权重衰减，L2正则化系数，用来减轻过拟合
    "loss_type": "infonce",  # 主实验：infonce；对照实验：bce
    "item_mapping_mode": "contiguous",  # contiguous: 连续映射；raw: 原始编号
    "infonce_temperature": 0.07, # InfoNCE 损失的温度系数。值越小，模型越强调正负样本之间的区分；值越大，分布更平缓
    "top_k": 20, # 召回阶段每个用户返回多少个候选物品
    "use_faiss": True, # 是否使用 FAISS 做近似最近邻召回
    "rebuild_faiss": False, # 是否强制重建 FAISS 索引（如果之前已经存在索引文件）
    # faiss 先分区，在查找
    "ann_nlist": 4096, # FAISS IVF 索引的聚类桶数。桶越多，检索更精细，但建索引和调参成本更高。
    "ann_nprobe": 32, # FAISS IVF 索引的搜索桶数。值越大，召回准确率越高，但查询速度越慢。
}

CONFIG["experiment_name"] = f"two_tower_{CONFIG['item_mapping_mode']}_{CONFIG['loss_type']}"
CONFIG["save_model_path"] = os.path.join(project_root, f"train/recall/result/{CONFIG['experiment_name']}_best.pt")
CONFIG["save_history_path"] = os.path.join(project_root, f"train/recall/result/{CONFIG['experiment_name']}_history.json")
CONFIG["save_recall_path"] = os.path.join(project_root, f"train/recall/result/{CONFIG['experiment_name']}_recall_results.json")
CONFIG["faiss_index_path"] = os.path.join(project_root, f"train/recall/result/{CONFIG['experiment_name']}_faiss_ivf.index")

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
print(f"FAISS 可用: {FAISS_AVAILABLE}")
print(f"当前实验: {CONFIG['experiment_name']}")
print(f"item 映射模式: {CONFIG['item_mapping_mode']}")
print(f"损失函数: {CONFIG['loss_type']}")

使用设备: cuda
FAISS 可用: True
当前实验: two_tower_contiguous_infonce
item 映射模式: contiguous
损失函数: infonce


In [4]:
# ===== 2) 数据集与 DataLoader =====
import importlib
from process import two_tower_data as two_tower_data_module

importlib.reload(two_tower_data_module) #强制刷新模块

from process.two_tower_data import (
    TwoTowerDataset,
    UserSample,
    build_user_samples,
    collate_fn,
)

In [5]:
# ===== 3) 召回引擎（公共逻辑） =====
class RecallEngine:
    """统一封装 Exact 与 FAISS ANN 召回。"""

    def __init__(
        self,
        model,
        device,
        dataset,
        use_faiss: bool = True,
        ann_nlist: int = 4096,
        ann_nprobe: int = 32,
        faiss_index_path: Optional[str] = None,
        rebuild_faiss: bool = False,
    ):
        self.model = model.to(device)
        self.device = device
        self.dataset = dataset
        self.use_faiss = use_faiss and FAISS_AVAILABLE
        self.ann_nlist = ann_nlist
        self.ann_nprobe = ann_nprobe
        self.faiss_index = None
        self.faiss_index_path = faiss_index_path
        self.rebuild_faiss = rebuild_faiss
        self.item_embeddings = None

        if use_faiss and not FAISS_AVAILABLE:
            print("FAISS 未安装，自动回退为精确全量打分召回。")

        if self.use_faiss:
            loaded = False
            if self.faiss_index_path and (not self.rebuild_faiss):
                loaded = self._load_faiss_index_if_exists()
            if not loaded:
                print("\n预计算物品 embedding（用于构建 FAISS 索引）...")
                self.item_embeddings = self._compute_all_item_embeddings()
                print(f"物品 embedding shape: {self.item_embeddings.shape}")
                self._build_faiss_index()
                if self.faiss_index_path:
                    self._save_faiss_index()
        else:
            print("\n预计算物品 embedding（用于精确召回）...")
            self.item_embeddings = self._compute_all_item_embeddings()
            print(f"物品 embedding shape: {self.item_embeddings.shape}")

    def _compute_all_item_embeddings(self):
        self.model.eval()
        all_item_ids = torch.arange(self.dataset.item_vocab_size, device=self.device)

        item_embeddings = []
        batch_size = 1024
        with torch.no_grad():
            for i in tqdm(range(0, len(all_item_ids), batch_size), desc="计算物品 embedding"):
                batch_items = all_item_ids[i : i + batch_size]
                batch_item_categories = torch.tensor(
                    self.dataset.item_category_by_id[
                        batch_items.detach().cpu().numpy()
                    ],
                    device=self.device,
                    dtype=torch.long,
                )
                batch_emb = self.model.get_item_embedding(
                    batch_items,
                    video_category_ids=batch_item_categories,
                )
                item_embeddings.append(batch_emb)

        return torch.cat(item_embeddings, dim=0)

    def _build_faiss_index(self):
        print("\n构建 FAISS ANN 索引...")
        item_emb_np = self.item_embeddings.detach().cpu().numpy().astype(np.float32)
        dim = item_emb_np.shape[1]

        faiss.normalize_L2(item_emb_np)
        quantizer = faiss.IndexFlatIP(dim)
        nlist = min(self.ann_nlist, max(1, item_emb_np.shape[0] // 100))

        self.faiss_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        self.faiss_index.train(item_emb_np)
        self.faiss_index.add(item_emb_np)
        self.faiss_index.nprobe = min(self.ann_nprobe, nlist)

        print(
            f"FAISS 索引完成: ntotal={self.faiss_index.ntotal}, "
            f"nlist={nlist}, nprobe={self.faiss_index.nprobe}"
        )

    def _load_faiss_index_if_exists(self):
        if not self.faiss_index_path or (not os.path.exists(self.faiss_index_path)):
            return False

        print(f"\n加载 FAISS 索引: {self.faiss_index_path}")
        index = faiss.read_index(self.faiss_index_path)

        if index.ntotal != self.dataset.item_vocab_size:
            print(
                "已保存索引与当前物品数不一致，"
                f"index.ntotal={index.ntotal}, item_vocab_size={self.dataset.item_vocab_size}，将重建索引。"
            )
            return False

        index.nprobe = min(self.ann_nprobe, index.nlist)
        self.faiss_index = index
        print(
            f"FAISS 索引加载完成: ntotal={self.faiss_index.ntotal}, "
            f"nlist={self.faiss_index.nlist}, nprobe={self.faiss_index.nprobe}"
        )
        return True

    def _save_faiss_index(self):
        os.makedirs(os.path.dirname(self.faiss_index_path), exist_ok=True)
        faiss.write_index(self.faiss_index, self.faiss_index_path)
        print(f"FAISS 索引已保存: {self.faiss_index_path}")

    def _search_with_faiss(self, user_emb: torch.Tensor, top_k: int):
        query = user_emb.detach().cpu().numpy().astype(np.float32)
        faiss.normalize_L2(query)

        distances, indices = self.faiss_index.search(query, top_k)
        item_ids = indices[0].tolist()
        scores = distances[0].tolist()
        return [(int(item_id), float(score)) for item_id, score in zip(item_ids, scores)]

    def recall_for_users(self, user_samples: Sequence[UserSample], top_k: int = 50):
        self.model.eval()
        recalls = {}

        recall_mode = "FAISS ANN" if self.faiss_index is not None else "Exact"
        print(f"\n为 {len(user_samples)} 个用户进行 Top-{top_k} 召回 (模式: {recall_mode})...")

        with torch.no_grad():
            for user_id, behavior_seq, behavior_mask, gender_id, age_id in tqdm(user_samples, desc="召回中"):
                user_id_tensor = torch.tensor([user_id], device=self.device)
                behavior_seq_tensor = torch.tensor(np.asarray([behavior_seq]), device=self.device)
                behavior_mask_tensor = torch.tensor(np.asarray([behavior_mask]), device=self.device)
                gender_tensor = torch.tensor([gender_id], device=self.device)
                age_tensor = torch.tensor([age_id], device=self.device)

                user_emb = self.model.get_user_embedding(
                    user_id_tensor,
                    behavior_seq_tensor,
                    behavior_mask_tensor,
                    gender_ids=gender_tensor,
                    age_ids=age_tensor,
                )

                if self.faiss_index is not None:
                    recalls[user_id] = self._search_with_faiss(
                        user_emb,
                        top_k=min(top_k, self.dataset.item_vocab_size),
                    )
                else:
                    scores = self.model.predict_batch(user_emb, self.item_embeddings)
                    top_scores, top_indices = torch.topk(scores, k=min(top_k, len(scores)))
                    recalls[user_id] = [
                        (int(item_id), float(score))
                        for item_id, score in zip(top_indices.cpu().numpy(), top_scores.cpu().numpy())
                    ]

        return recalls

In [6]:
# ===== 4) 训练器定义 =====
class TwoTowerTrainer:
    """双塔召回模型训练器。"""

    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        device,
        learning_rate=0.001,
        weight_decay=1e-5,
        loss_type="bce",
        infonce_temperature=0.07,
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.loss_type = loss_type.lower()
        self.use_infonce = self.loss_type == "infonce"
        self.infonce_temperature = infonce_temperature

        self.optimizer = optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )

        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode="min",
            factor=0.5,
            patience=2,
        )

        self.criterion = nn.BCEWithLogitsLoss()

        self.history = {
            "train_loss": [],
            "val_loss": [],
            "val_auc": [],
            "val_precision": [],
            "val_recall": [],
        }

    def _compute_infonce_loss(self, user_repr, item_repr, labels):
        positive_mask = labels > 0.5
        if positive_mask.sum().item() < 2:
            return None

        pos_user_repr = user_repr[positive_mask]
        pos_item_repr = item_repr[positive_mask]
        logits = torch.matmul(pos_user_repr, pos_item_repr.t()) / self.infonce_temperature
        targets = torch.arange(pos_user_repr.size(0), device=logits.device)

        loss_user_to_item = nn.functional.cross_entropy(logits, targets)
        loss_item_to_user = nn.functional.cross_entropy(logits.t(), targets)
        return 0.5 * (loss_user_to_item + loss_item_to_user)

    def _forward_batch(self, batch_data):
        (
            user_ids,
            item_ids,
            behavior_sequences,
            behavior_masks,
            gender_ids,
            age_ids,
            video_category_ids,
            labels,
        ) = [x.to(self.device) for x in batch_data]

        scores, embeddings = self.model(
            user_ids,
            item_ids,
            behavior_sequences,
            behavior_masks,
            gender_ids=gender_ids,
            age_ids=age_ids,
            video_category_ids=video_category_ids,
            return_embeddings=self.use_infonce,
        )
        return scores, embeddings, labels

    def _compute_loss(self, scores, labels, embeddings=None):
        if self.use_infonce:
            if embeddings is None:
                raise ValueError("InfoNCE 训练需要返回 embeddings")
            return self._compute_infonce_loss(
                embeddings["user_repr"],
                embeddings["item_repr"],
                labels,
            )

        return self.criterion(scores, labels)

    def train_epoch(self):
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        skipped_batches = 0

        pbar = tqdm(self.train_loader, desc="Training")
        for batch_data in pbar:
            scores, embeddings, labels = self._forward_batch(batch_data)
            loss = self._compute_loss(scores, labels, embeddings)
            if loss is None:
                skipped_batches += 1
                continue

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
            self.optimizer.step()

            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / max(num_batches, 1)
        if skipped_batches > 0:
            print(f"InfoNCE 训练中跳过了 {skipped_batches} 个正样本不足的 batch")
        return avg_loss

    def validate(self):
        self.model.eval()
        total_loss = 0.0
        num_batches = 0
        skipped_batches = 0
        all_scores = []
        all_labels = []

        with torch.no_grad():
            for batch_data in tqdm(self.val_loader, desc="Validating"):
                scores, embeddings, labels = self._forward_batch(batch_data)

                loss = self._compute_loss(scores, labels, embeddings)
                if loss is None:
                    skipped_batches += 1
                    all_scores.extend(torch.sigmoid(scores).cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
                    continue
                total_loss += loss.item()
                num_batches += 1

                all_scores.extend(torch.sigmoid(scores).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_loss = total_loss / max(num_batches, 1)
        all_scores = np.array(all_scores)
        all_labels = np.array(all_labels)
        binary_preds = (all_scores >= 0.5).astype(int)

        try:
            auc = roc_auc_score(all_labels, all_scores)
        except Exception:
            auc = 0.5

        precision = precision_score(all_labels, binary_preds, zero_division=0)
        recall = recall_score(all_labels, binary_preds, zero_division=0)

        if skipped_batches > 0:
            print(f"InfoNCE 验证中跳过了 {skipped_batches} 个正样本不足的 batch")

        return avg_loss, auc, precision, recall

    def train(self, num_epochs, save_path):
        best_val_loss = float("inf")

        print(f"\n开始训练 {num_epochs} 个 epochs...")
        print(f"训练样本数: {len(self.train_loader.dataset)}")
        print(f"验证样本数: {len(self.val_loader.dataset)}")

        for epoch in range(num_epochs):
            print("\n" + "=" * 60)
            print(f"Epoch {epoch + 1}/{num_epochs}")
            print("=" * 60)

            train_loss = self.train_epoch()
            val_loss, val_auc, val_precision, val_recall = self.validate()
            self.scheduler.step(val_loss)

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["val_auc"].append(val_auc)
            self.history["val_precision"].append(val_precision)
            self.history["val_recall"].append(val_recall)

            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": self.model.state_dict(),
                        "optimizer_state_dict": self.optimizer.state_dict(),
                        "val_loss": val_loss,
                        "val_auc": val_auc,
                        "val_precision": val_precision,
                        "val_recall": val_recall,
                    },
                    save_path,
                )
                print(f"✓ 保存最佳模型到 {save_path}")

        print("\n训练完成!")
        print(f"最佳验证 Loss: {best_val_loss:.4f}")
        return self.history

In [7]:
# ===== 5) 构建数据与模型 =====
import importlib
from process import two_tower_data as two_tower_data_module

importlib.reload(two_tower_data_module) #再次刷新模块，避免内核里残留旧版类定义
TwoTowerDataset = two_tower_data_module.TwoTowerDataset
collate_fn = two_tower_data_module.collate_fn
build_user_samples = two_tower_data_module.build_user_samples

dataset = TwoTowerDataset(
    CONFIG["data_path"],
    max_seq_len=CONFIG["max_seq_len"],
    item_mapping_mode=CONFIG["item_mapping_mode"],
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

model_config = {
    "user_vocab_size": dataset.user_vocab_size,
    "item_vocab_size": dataset.item_vocab_size,
    "gender_vocab_size": dataset.gender_vocab_size,
    "age_vocab_size": dataset.age_vocab_size,
    "video_category_vocab_size": dataset.video_category_vocab_size,
    "embedding_dim": 32,
    "transformer_heads": 2,
    "transformer_layers": 1,
    "user_tower_dims": [128, 64],
    "item_tower_dims": [128, 64],
    "output_dim": 32,
    "dropout": 0.2,
    "max_seq_len": CONFIG["max_seq_len"],
    "temperature": 0.05,
}

model = build_two_tower_model(**model_config)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

正在加载数据: d:\Machine_learning\Recommend\LHY\data/ctr_data_500k.csv
数据 shape: (500000, 20)
用户数: 3764
物品数(去重后): 178108
item 映射模式: contiguous
样本数: 500000
User vocab size: 3767
Item vocab size: 178109
Gender vocab size: 3
Age vocab size: 8
Video-category vocab size: 3
正样本比例: 0.2742
模型参数量: 11,579,136


In [8]:
# ===== 6) 训练并保存 =====
trainer = TwoTowerTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    loss_type=CONFIG["loss_type"],
    infonce_temperature=CONFIG["infonce_temperature"],
)

history = trainer.train(num_epochs=CONFIG["num_epochs"], save_path=CONFIG["save_model_path"])

os.makedirs(os.path.dirname(CONFIG["save_history_path"]), exist_ok=True)
with open(CONFIG["save_history_path"], "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2, ensure_ascii=False)

print(f"训练历史已保存到: {CONFIG['save_history_path']}")


开始训练 6 个 epochs...
训练样本数: 400000
验证样本数: 100000

Epoch 1/6


Training:  86%|████████▌ | 2695/3125 [01:20<00:12, 33.34it/s, loss=3.6613]


KeyboardInterrupt: 

In [ ]:
# ===== 7) 加载最佳模型并召回 =====
checkpoint = torch.load(CONFIG["save_model_path"], weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

print(
    f"加载最佳模型 (epoch {checkpoint['epoch'] + 1}," #保存的 epoch 是从 0 开始计数的“下标”
    f"val_auc: {checkpoint['val_auc']:.4f}, "
    f"val_precision: {checkpoint.get('val_precision', 0.0):.4f}, "
    f"val_recall: {checkpoint.get('val_recall', 0.0):.4f})"
)

recall_engine = RecallEngine(
    model=model,
    device=device,
    dataset=dataset,
    use_faiss=CONFIG["use_faiss"],
    ann_nlist=CONFIG["ann_nlist"],
    ann_nprobe=CONFIG["ann_nprobe"],
    faiss_index_path=CONFIG["faiss_index_path"],
    rebuild_faiss=CONFIG["rebuild_faiss"],
)

unique_users = np.unique(dataset.user_ids)
sample_users = unique_users[:20]
user_samples = build_user_samples(dataset, user_ids=sample_users, max_seq_len=CONFIG["max_seq_len"])

recalls = recall_engine.recall_for_users(user_samples, top_k=CONFIG["top_k"])

recall_results = {}
for user_id, items in recalls.items():
    recall_results[int(user_id)] = {
        "recalled_items": [int(item[0]) for item in items],
        "scores": [float(item[1]) for item in items],
    }

os.makedirs(os.path.dirname(CONFIG["save_recall_path"]), exist_ok=True)
with open(CONFIG["save_recall_path"], "w", encoding="utf-8") as f:
    json.dump(recall_results, f, indent=2, ensure_ascii=False)

print(f"召回结果已保存到: {CONFIG['save_recall_path']}")
print(f"共为 {len(recall_results)} 个用户生成了召回候选集")

加载最佳模型 (epoch 2,val_auc: 0.7316, val_precision: 0.5918, val_recall: 0.3133)

预计算物品 embedding（用于构建 FAISS 索引）...


计算物品 embedding: 100%|██████████| 174/174 [00:00<00:00, 2453.26it/s]

物品 embedding shape: torch.Size([178109, 32])

构建 FAISS ANN 索引...


FAISS 索引完成: ntotal=178109, nlist=1781, nprobe=32
FAISS 索引已保存: /root/autodl-tmp/LHY/train/recall/result/two_tower_contiguous_bce_faiss_ivf.index

为 20 个用户进行 Top-20 召回 (模式: FAISS ANN)...


召回中: 100%|██████████| 20/20 [00:00<00:00, 448.83it/s]

召回结果已保存到: /root/autodl-tmp/LHY/train/recall/result/two_tower_contiguous_bce_recall_results.json
共为 20 个用户生成了召回候选集


In [ ]:
# ===== 8) 结果预览 =====
print("召回结果示例（前 3 个用户）：")
for user_id in list(recalls.keys())[:3]:
    items = recalls[user_id]
    print(f"\n用户 {user_id}:")
    print(f"  召回 Top-10 物品: {[item[0] for item in items[:10]]}")
    print(f"  对应分数: {[f'{item[1]:.4f}' for item in items[:10]]}")

召回结果示例（前 3 个用户）：

用户 1:
  召回 Top-10 物品: [170407, 136994, 170391, 136133, 136028, 137236, 136895, 136320, 5005, 2327]
  对应分数: ['0.0678', '0.0663', '0.0654', '0.0651', '0.0648', '0.0648', '0.0647', '0.0646', '0.0646', '0.0645']

用户 2:
  召回 Top-10 物品: [170407, 136994, 170391, 136133, 340, 779, 136028, 136900, 136879, 5005]
  对应分数: ['-0.0162', '-0.0178', '-0.0191', '-0.0191', '-0.0194', '-0.0196', '-0.0196', '-0.0196', '-0.0199', '-0.0199']

用户 3:
  召回 Top-10 物品: [170407, 136994, 137236, 170391, 136028, 136320, 136895, 136133, 2327, 140390]
  对应分数: ['0.0772', '0.0759', '0.0748', '0.0748', '0.0745', '0.0745', '0.0743', '0.0743', '0.0743', '0.0741']


In [ ]:
# ===== 9) 四组实验结果对比 =====
from pathlib import Path

compare_configs = [
    ("原始 BCE", "two_tower_raw_bce_history.json"),
    ("原始 InfoNCE", "two_tower_raw_infonce_history.json"),
    ("连续映射 BCE", "two_tower_contiguous_bce_history.json"),
    ("连续映射 InfoNCE", "two_tower_contiguous_infonce_history.json"),
]

summary_rows = []
for name, filename in compare_configs:
    history_path = Path(project_root) / "train/recall/result" / filename
    if not history_path.exists():
        summary_rows.append({
            "experiment": name,
            "history_file": filename,
            "status": "missing",
        })
        continue

    with open(history_path, "r", encoding="utf-8") as f:
        history_data = json.load(f)

    val_auc_list = history_data.get("val_auc", [])
    val_loss_list = history_data.get("val_loss", [])
    if val_auc_list:
        best_epoch = int(np.argmax(val_auc_list)) + 1
        best_val_auc = float(np.max(val_auc_list))
    else:
        best_epoch = None
        best_val_auc = None

    best_val_loss = float(np.min(val_loss_list)) if val_loss_list else None
    summary_rows.append({
        "experiment": name,
        "history_file": filename,
        "status": "ok",
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
        "best_val_loss": best_val_loss,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


  experiment                              history_file status  best_epoch  best_val_auc  best_val_loss
      原始 BCE            two_tower_raw_bce_history.json     ok           2      0.733228       0.517552
  原始 InfoNCE        two_tower_raw_infonce_history.json     ok           6      0.546523       3.384333
    连续映射 BCE     two_tower_contiguous_bce_history.json     ok           2      0.731631       0.517349
连续映射 InfoNCE two_tower_contiguous_infonce_history.json     ok           5      0.548078       3.393381
